In [5]:
from google.colab import drive
drive.mount('/content/drive')
import os
import numpy as np
import pandas as pd
import polars as pl

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Define the full dataset file path
file_path = "/content/drive/MyDrive/Colab Notebooks/aml-compliance-system-main/Data/raw/HI-Large_Trans.csv"

# Read the data using lazy evaluation (scan_csv) to prevent RAM crashes on massive datasets
q = pl.scan_csv(file_path)

# Verify the total row count
print("Total rows:", q.select(pl.len()).collect().item())

Total rows: 179702229


In [7]:
# Convert Timestamp flexibly and extract features correctly
q_transformed = q.with_columns(
    pl.col("Timestamp").str.to_datetime(strict=False).alias("Timestamp")
).with_columns([
    pl.col("Timestamp").dt.hour().alias("Transaction_Hour"),
    pl.col("Timestamp").dt.weekday().alias("Day_Of_Week"),
    pl.col("Timestamp").dt.month().alias("Month"),
    pl.col("Timestamp").dt.weekday().is_in([4, 5]).cast(pl.Int32).alias("Is_Weekend")
])

# Preview the result to ensure values are correct
print(q_transformed.select(["Timestamp", "Transaction_Hour", "Day_Of_Week", "Month", "Is_Weekend"]).head(3).collect())


shape: (3, 5)
┌─────────────────────┬──────────────────┬─────────────┬───────┬────────────┐
│ Timestamp           ┆ Transaction_Hour ┆ Day_Of_Week ┆ Month ┆ Is_Weekend │
│ ---                 ┆ ---              ┆ ---         ┆ ---   ┆ ---        │
│ datetime[μs]        ┆ i8               ┆ i8          ┆ i8    ┆ i32        │
╞═════════════════════╪══════════════════╪═════════════╪═══════╪════════════╡
│ 2022-08-01 00:17:00 ┆ 0                ┆ 1           ┆ 8     ┆ 0          │
│ 2022-08-01 00:02:00 ┆ 0                ┆ 1           ┆ 8     ┆ 0          │
│ 2022-08-01 00:17:00 ┆ 0                ┆ 1           ┆ 8     ┆ 0          │
└─────────────────────┴──────────────────┴─────────────┴───────┴────────────┘


In [10]:


file_path = "/content/drive/MyDrive/Colab Notebooks/aml-compliance-system-main/Data/raw/HI-Large_Trans.csv"
output_dir = "/content/drive/MyDrive/Colab Notebooks/aml-compliance-system-main/Data/processed"
output_path = os.path.join(output_dir, "HI-Large_Trans_with_features.csv")

os.makedirs(output_dir, exist_ok=True)

chunk_size = 500_000
first_chunk = True

print("Processing dataset in chunks...")
for chunk in pd.read_csv(file_path, chunksize=chunk_size):
    chunk['Timestamp'] = pd.to_datetime(chunk['Timestamp'], errors='coerce')
    chunk['Transaction_Hour'] = chunk['Timestamp'].dt.hour
    chunk['Day_Of_Week'] = chunk['Timestamp'].dt.dayofweek
    chunk['Month'] = chunk['Timestamp'].dt.month
    chunk['Is_Weekend'] = chunk['Timestamp'].dt.dayofweek.isin([4, 5]).astype(int)

    chunk.to_csv(output_path, mode='w' if first_chunk else 'a', index=False, header=first_chunk)
    first_chunk = False

print("Done! File processed and saved successfully.")


Processing dataset in chunks...
Done! File processed and saved successfully.


In [12]:


file_path = "/content/drive/MyDrive/Colab Notebooks/aml-compliance-system-main/Data/raw/HI-Large_Trans.csv"
output_dir = "/content/drive/MyDrive/Colab Notebooks/aml-compliance-system-main/Data/processed/chunks"

# Ensure the chunks directory exists on Google Drive
os.makedirs(output_dir, exist_ok=True)

chunk_size = 1_000_000
print("Building Phase 2 - Section A in separate chunk files...")

for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size)):
    timestamp = pd.to_datetime(chunk['Timestamp'], errors='coerce')

    features_df = pd.DataFrame({
        'Transaction_Hour': timestamp.dt.hour,
        'Day_Of_Week': timestamp.dt.dayofweek,
        'Month': timestamp.dt.month,
        'Is_Weekend': timestamp.dt.dayofweek.isin([4, 5]).astype(int),
        'Log_Amount_Paid': np.log1p(chunk['Amount Paid']),
        'Log_Amount_Received': np.log1p(chunk['Amount Received']),
        'Amount_Difference': chunk['Amount Paid'] - chunk['Amount Received'],
        'Amount_Ratio': chunk['Amount Paid'] / (chunk['Amount Received'] + 1)
    })

    # Save each chunk as a separate file to prevent local caching/disk full errors
    chunk_filename = os.path.join(output_dir, f"features_chunk_{i}.csv")
    features_df.to_csv(chunk_filename, index=False)
    print(f"Saved chunk {i} successfully.")

print("All chunks successfully saved to your Drive!")


Building Phase 2 - Section A in separate chunk files...
Saved chunk 0 successfully.
Saved chunk 1 successfully.
Saved chunk 2 successfully.
Saved chunk 3 successfully.
Saved chunk 4 successfully.
Saved chunk 5 successfully.
Saved chunk 6 successfully.
Saved chunk 7 successfully.
Saved chunk 8 successfully.
Saved chunk 9 successfully.
Saved chunk 10 successfully.
Saved chunk 11 successfully.
Saved chunk 12 successfully.
Saved chunk 13 successfully.
Saved chunk 14 successfully.
Saved chunk 15 successfully.
Saved chunk 16 successfully.
Saved chunk 17 successfully.
Saved chunk 18 successfully.
Saved chunk 19 successfully.
Saved chunk 20 successfully.
Saved chunk 21 successfully.
Saved chunk 22 successfully.
Saved chunk 23 successfully.
Saved chunk 24 successfully.
Saved chunk 25 successfully.
Saved chunk 26 successfully.
Saved chunk 27 successfully.
Saved chunk 28 successfully.
Saved chunk 29 successfully.
Saved chunk 30 successfully.
Saved chunk 31 successfully.
Saved chunk 32 successfull

KeyboardInterrupt: 

In [ ]:
import os

output_dir = "/content/drive/MyDrive/Colab Notebooks/aml-compliance-system-main/Data/processed/chunks"

print("Checking processed chunks for Section A...")
chunk_files = sorted([os.path.join(output_dir, f) for f in os.listdir(output_dir) if f.startswith("features_chunk_")])

print(f"Total ready chunks found: {len(chunk_files)}")
print("Section A is fully processed and safely stored in chunks on your Drive. Ready for Section B!")
